# Intro to pandas

This notebook has been set up to cover some introductory concepts in pandas for CCC colleagues.

### Packages

A reasonable amount of functionality comes built in to Python, covering everything we did in the 'intro to python' session and more. However, one of the great things about Python (and other languages, like R (and KNIME)) is that they have an active user base who contribute packages. Packages are add-on pieces of functionality which have been designed to help with particular (or quite general) uses. 

### Pandas

One of the most popular packages that we will certainly be using in our work is pandas, which provides a range of functionality for manipulating tabular data. Let's take a look.

In [ ]:
# installing the pandas package

!pip install pandas

# installing another package which we will need later on

!pip install openpyxl

In [ ]:
# now importing it into this notebook so we can use it

import pandas as pd

The central concept in pandas is something called a DataFrame, which is basically a data table with a set of rows and columns, and a lot of integrated functionality. DataFrames can be created by reading in files (e.g. from Excel), which is what we're going to do to explore some basic functionality here.

We've got the cb7 full dataset saved in a data subfolder within this repository. Let's take a look at that using pandas.

In [ ]:
# we know our file has multiple tabs
# note use of pd. syntax, which indicates that we are using some pandas functionality

file = pd.ExcelFile("../data/cb7_full_dataset.xlsx")
file.sheet_names

In [ ]:
# we don't want to look at all of these tabs now - let's just look at the sector-level data
# note that the term df is commonly used as a shorthand for DataFrame

df = pd.read_excel(file, sheet_name="Sector-level data")

### Understanding dataframes

Before we do anything at all, we need to understand some basics about a dataframe's structure. A dataframe has the following key components:

1) Rows
2) Columns
3) An index

The first should be intuitive to anyone who has used Excel. The third might not be, but is important to understand for many operations in pandas.

In [ ]:
# we've now got a dataframe
# Let's look at the first few rows to confirm it is what we want

df.head(5)

In [ ]:
# looking at the column names

df.columns

In [ ]:
# looking at an individual column
# this is actually a pandas Series

df["scenario"]

In [ ]:
# looking at the index

df.index

Having a range index like this is probably not preferable for timeseries data, where we would generally want our time/date column to be the index for simplicity. We will address this later.

In [ ]:
# we can also use some very general pandas methods to describe our dataset
# first the .info method, which gives us an overview

df.info()

In [ ]:
# we can use .describe to give us basic stats about the numeric columns

df.describe()

In [ ]:
# that looks about right
# we can use some other basic functionality to inspect individual columns
# note use of square brackets to access individual columns

print(f"The data covers the following years: {df["year"].unique()}") # looking at the year-range

In [ ]:
# we can also use a loop to look at main entries in each columns:

for column in df.columns:
    print(column)
    print(df[column].unique())

### Simple operations on dataframes

The above operations give us a pretty good idea of what is in our data. We can now look at doing some basic operations to transform it.

Let's start with filtering data, in the same way that you might in Excel.

In [ ]:
# let's also save the results to a new dataframe to carry forward

cost_df = df.loc[df["variable"].str.contains("Cost")] # using .loc functionality to filter to only cost data

In [ ]:
# let's inspect this new dataframe

cost_df

In [ ]:
# let's look at what the values in this column are
# we might want to narrow this down more

cost_df["variable"].unique()

We probably don't want both adjusted and unadjusted figures if we're going to do any analysis in a grouped way. Let's remove the unadjusted figures.


In [ ]:
# using .loc again - note use of the ~ to indicate the inverse of our condition

cost_df = cost_df.loc[~cost_df["variable"].str.contains("unadjusted")]

cost_df["variable"].unique()

We might want to change some of the data e.g. simplifying variable names.

In [ ]:
# defining a renaming dictionary to replace the current entries with

renamer = {'Cost: additional capital expenditure (adjusted)' : 'capex',
           'Cost: additional operating expenditure (adjusted)' : 'opex',
           'Cost: additional capital expenditure annualised (adjusted)' : 'annualised_capex'}

cost_df["variable"] = cost_df["variable"].replace(renamer)

cost_df

### More advanced operations

One thing we will probably often want to do to data is group it in some way (analogous to SUMIF, AVERAGEIF etc in Excel). Let's create a new dataframe which stores the results of a grouping operation.

In [ ]:
# Note the specific syntax here: selecting columns, .groupby(), .sum()

grouped_costs_df = cost_df[["year", "variable", "value"]].groupby(by=["year", "variable"]).sum()

grouped_costs_df

Another key operation is reshaping our data. If we have long data (as we do here), we might want to pivot it to get the entries of one column as new columns. The inverse operation also exists, using the .melt() method in pandas.

In [ ]:
# we can also do pivot operations to change the shape of the data
# NOTE: WE'RE SETTING AN INDEX IN THE BELOW OPERATION. THIS WILL BE RELEVANT LATER ON.

pivot_df = grouped_costs_df.reset_index().pivot(columns="variable", values="value", index="year")

pivot_df

In [ ]:
# we can sort the data like you would in excel if that's of interest

pivot_df.sort_values(by="capex", ascending=False)

In [ ]:
# note that sorting operation has not been saved!
# 

pivot_df

A fundamental operation we haven't explored yet - creating new columns!

In [ ]:
# note syntax - we create a new column simply by writing its name and assigning results

pivot_df["net_annual_cost"] = pivot_df["annualised_capex"] + pivot_df["opex"] 

In [ ]:
# looking at that again

pivot_df

In [ ]:
# we might want to categorise a year as saving/non-saving

pivot_df["saving"] = pivot_df["net_annual_cost"].apply(lambda x: True if x < 0 else False)
pivot_df

We don't have to do everything in columns within our dataframe. Let's save some variables for key statistics.

In [ ]:
# we can pick out the years with the highest and lowest net cost

max_spend = pivot_df["net_annual_cost"].max()
max_spend_year = pivot_df[pivot_df["net_annual_cost"] == max_spend].index.values[0]

# and total cost across entire period

total_cost = pivot_df["net_annual_cost"].sum()

print(f"""The year with highest net cost is {max_spend_year}, with costs of £m{max_spend:,.0f}.
The net total cost is £m {total_cost:,.0f}""")

### Joining and merging

This is all great, but what happens when we have another dataset that we want to bring in to our analysis. Joining and merging datasets is a key operation that we will encounter a lot when working with data.

Let's take a look at the GDP forecast dataset for our example.

In [ ]:
# we have another dataset, which is the gdp forecast to 2050 used in CB7
# let's read this in and then do some more operations

gdp_df = pd.read_csv("../data/uk_gdp_forecast.csv")

gdp_df

We can see that this looks about right for the dataframe that we have been working with. The GDP dataframe has a column with values, and another with years. We should be able to join these together fairly easily right?

In [ ]:
# let's do a join operation so that we can get this in a dataframe with our other data

new_df = pivot_df.join(gdp_df)

We've got an error. Why?

Because our two dataframes have different indexes, so they won't join automatically.

In [ ]:
# that hasn't worked! Why not?
# the issue is to do with indexes

print(pivot_df.index)
print(gdp_df.index)

In [ ]:
# let's set the gdp_df index to year and try again

gdp_df = gdp_df.set_index("year")

new_df = pivot_df.join(gdp_df)

In [ ]:
new_df

You might notice another particularity in our joined dataframe. Some of the years from the GDP dataset are missing. This is because of the type of join we're using, which is a 'left' join by default. Let's try another type and see what we get.

In [ ]:
# looking at the below, we see that the data is only shown from 2025 onwards, even thought the uk_gdp series starts in 2021
# the type of join we do here is key!

new_df = pivot_df.join(gdp_df, how="outer")

new_df

And then inversing the order of the join, but with yet another type.

In [ ]:
new_df = gdp_df.join(pivot_df, how="right")

In [ ]:
new_df

In the end, it's probably best to just have the data for 2025 onwards, so let's keep that.

### Bonus: graphs

There are lots of libraries in python to visualise data very nicely. There is a slight learning curve to making plots look nice in many libraries though, which we won't go into right now. However, using basic plotting can be very helpful for sense checking our data. Let's do that now.

In [ ]:
!pip install matplotlib

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9,6))

ax.plot(new_df["opex"], label="Operating expenditure")
ax.plot(new_df["annualised_capex"], label="Annualised capital expenditure")
ax.plot(new_df["net_annual_cost"], label="Net annual cost")
ax.set_ylabel("Cost (£m)")
ax.axhline(y=0, color="black")
ax.legend()

plt.show()